# MongoDB Landing Zone — Exploration

Verbindung: **MongoDB Atlas** (`mongo-mk1`, eu-central-1 Frankfurt)  
Datenbank: `airline_landing`  
User: `airline-reader-ro` (read-only) via `MONGO_URI` aus `.env`

| Collection | Befüllt durch | Inhalt |
|---|---|---|
| `adsb_raw` | `collectors/adsb_collector.py` | ADS-B Snapshots (adsb.lol, Berlin 60 nm Radius) |
| `opensky_raw` | `collectors/opensky_collector.py` | Departures & Arrivals pro Airport (OpenSky Network) |
| `flight_tracker_raw` | `collectors/flight_tracker.py` | Einzelne Flugzeugpositionen (getracked) |

**Sektionen 1–8:** ADS-B Exploration  
**Sektionen 9–11:** OpenSky Exploration + Cross-Collection Join  
**Sektionen 12–14:** flight_tracker_raw Exploration

In [1]:
import sys
from pathlib import Path
import pandas as pd
from datetime import timezone

sys.path.insert(0, '.')
from db.mongo.connector import from_env

db = from_env().connect()
print("Verbunden mit MongoDB")

Verbunden mit MongoDB


## 1. Collections und Dokumentenanzahl

In [2]:
client = db._client
database = client["airline_landing"]

print("Collections in airline_landing:\n")
for name in sorted(database.list_collection_names()):
    count = database[name].count_documents({})
    latest = database[name].find_one({}, {"collected_at": 1, "_id": 0}, sort=[("collected_at", -1)])
    latest_ts = latest.get("collected_at", "?")[:19] if latest else "?"
    print(f"  {name:<20}  docs={count:>5}  latest={latest_ts}")

col = db.collection("adsb_raw")

Collections in airline_landing:

  adsb_raw              docs=    4  latest=2026-05-27T16:20:13
  flight_tracker_raw    docs=  116  latest=2026-05-20T19:35:37
  opensky_raw           docs=    4  latest=2026-05-18T19:25:21


## 2. Neueste Snapshots (ohne `ac`-Array)

In [3]:
recent = list(col.find({}, {'ac': 0}).sort('collected_at', -1).limit(10))

df_snaps = pd.DataFrame([
    {
        'collected_at': d['collected_at'],
        'total': d['total'],
        'source': d['source'],
        '_id': str(d['_id']),
    }
    for d in recent
])
print(df_snaps.to_string(index=False))

                    collected_at  total   source                      _id
2026-05-27T16:20:13.225953+00:00     31 adsb.lol 6a1719bdafa5a300809d9fdd
2026-05-18T14:18:59.387774+00:00     29 adsb.lol 6a0b1ff569048342fb37eabc
2026-05-18T11:41:54.769203+00:00     29 adsb.lol 6a0afb029150ac72978f4310
2026-05-18T11:31:21.576214+00:00     35 adsb.lol 6a0af889dad6ae075993469c


## 3. Letzten Snapshot als DataFrame aufklappen

In [4]:
latest = col.find_one({}, sort=[('collected_at', -1)])
print(f"Snapshot: {latest['collected_at']}  —  {latest['total']} aircraft")

df = pd.DataFrame(latest['ac'])
print(f"Shape: {df.shape}")
print(f"Spalten: {list(df.columns)}")

Snapshot: 2026-05-27T16:20:13.225953+00:00  —  31 aircraft
Shape: (31, 54)
Spalten: ['hex', 'type', 'flight', 'r', 't', 'alt_baro', 'alt_geom', 'gs', 'ias', 'tas', 'mach', 'wd', 'ws', 'oat', 'tat', 'track', 'roll', 'mag_heading', 'true_heading', 'baro_rate', 'squawk', 'emergency', 'category', 'nav_qnh', 'nav_altitude_mcp', 'nav_heading', 'nav_modes', 'lat', 'lon', 'nic', 'rc', 'seen_pos', 'version', 'nic_baro', 'nac_p', 'nac_v', 'sil', 'sil_type', 'gva', 'sda', 'alert', 'spi', 'mlat', 'tisb', 'messages', 'seen', 'rssi', 'dst', 'dir', 'track_rate', 'geom_rate', 'nav_altitude_fms', 'dbFlags', 'calc_track']


In [5]:
# Lesbare Übersicht
cols = [c for c in ['hex', 'flight', 'r', 't', 'alt_baro', 'gs', 'lat', 'lon', 'dst'] if c in df.columns]
print(df[cols].to_string(index=False))

   hex   flight      r    t alt_baro    gs       lat       lon    dst
3e2a09 QGA50P   D-IHCR E50P    19550 406.0 52.552868 11.816345 58.104
3c56f2 EWG45B   D-AEWR A320    27800 396.9 52.071564 12.245204 50.475
3c0936 SDR83EF  D-ASMR A320    16475 357.0 52.561752 12.311554 40.100
484557 KLM1780  PH-BXZ B738    20400 404.2 52.298584 12.427979 38.237
4068f6 EXS6VN   G-GDFR B738    37000 515.8 53.254643 12.604388 52.895
48af01 LOT407   SP-LVB B38M    38000 400.3 52.022126 12.698704 39.647
39de55 TVF89NN     NaN  NaN    16275 326.1 52.253632 12.921600 23.939
486741 KLM67B   PH-AXC A21N    12350 422.0 52.557942 12.947780 16.995
3d2834 DENMA    D-ENMA DA40     1700 100.0 52.528893 13.232112  6.506
4aca8a NSZ2902  SE-RTJ B38M    38000 452.4 52.467770 13.233268  7.116
48c2ae RYR89YH  SP-RZO B38M    36000 389.7 52.961503 13.290318 26.876
502d24 AUA22S   YL-AAV BCS3    13125 370.0 52.088928 13.386078 25.874
3d4893 DEZST    D-EZST SR22     6500 164.4 52.766281 13.395996 14.812
3c6521 LHX4EL   D-AI

## 4. Datenqualität — Vollständigkeit der Kernfelder

In [6]:
key_fields = ['hex', 'flight', 'lat', 'lon', 'alt_baro', 'gs', 'r', 't']
present = {f: df[f].notna().sum() if f in df.columns else 0 for f in key_fields}
for field, count in present.items():
    pct = count / len(df) * 100
    print(f"  {field:<12} {count:>3}/{len(df)}  ({pct:.0f}%)")

  hex           31/31  (100%)
  flight        31/31  (100%)
  lat           31/31  (100%)
  lon           31/31  (100%)
  alt_baro      31/31  (100%)
  gs            30/31  (97%)
  r             28/31  (90%)
  t             28/31  (90%)


## 5. alt_baro: Zahl vs. 'ground'

In [7]:
if 'alt_baro' in df.columns:
    on_ground = (df['alt_baro'] == 'ground').sum()
    airborne  = (df['alt_baro'] != 'ground').sum()
    print(f"Am Boden:    {on_ground}")
    print(f"In der Luft: {airborne}")

    df['alt_baro_ft'] = pd.to_numeric(df['alt_baro'], errors='coerce')
    df['on_ground']   = df['alt_baro'] == 'ground'

Am Boden:    7
In der Luft: 24


## 6. Top-Callsigns über Berlin

In [8]:
if 'flight' in df.columns:
    top = (
        df['flight']
        .str.strip()
        .replace('', pd.NA)
        .dropna()
        .value_counts()
        .head(20)
    )
    print(top.to_string())

flight
QGA50P     1
EWG45B     1
SDR83EF    1
KLM1780    1
EXS6VN     1
LOT407     1
TVF89NN    1
KLM67B     1
DENMA      1
NSZ2902    1
RYR89YH    1
AUA22S     1
DEZST      1
LHX4EL     1
N926QS     1
BAW8RM     1
DLH4MA     1
EJU82DY    1
TKJ9HJ     1
RYR4JQ     1


## 7. Alle Snapshots zusammenführen — Zeitreihe der Flugzeugzahl

In [9]:
all_snaps = list(col.find({}, {'ac': 0}).sort('collected_at', 1))

df_ts = pd.DataFrame([
    {'collected_at': d['collected_at'], 'total': d['total']}
    for d in all_snaps
])
df_ts['collected_at'] = pd.to_datetime(df_ts['collected_at'], utc=True)

print(df_ts.to_string(index=False))
print(f"\nSchnitt: {df_ts['total'].mean():.1f} aircraft/snapshot")
print(f"Min/Max: {df_ts['total'].min()} / {df_ts['total'].max()}")

                    collected_at  total
2026-05-18 11:31:21.576214+00:00     35
2026-05-18 11:41:54.769203+00:00     29
2026-05-18 14:18:59.387774+00:00     29
2026-05-27 16:20:13.225953+00:00     31

Schnitt: 31.0 aircraft/snapshot
Min/Max: 29 / 35


---
## 9. OpenSky Raw — Überblick

Jedes Dokument = ein API-Call (departures **oder** arrivals) für einen Airport und ein Zeitfenster.

In [10]:
from datetime import datetime, timezone

osky_col = db.collection("opensky_raw")

docs = list(osky_col.find({}, {"flights": 0}).sort("collected_at", -1))

df_osky = pd.DataFrame([
    {
        "collected_at": d["collected_at"][:19],
        "endpoint":     d["endpoint"],
        "airport":      d["query_params"]["airport"],
        "flight_count": d["flight_count"],
        "window_h":     round((d["query_params"]["end"] - d["query_params"]["begin"]) / 3600),
        "_id":          str(d["_id"]),
    }
    for d in docs
])

print(f"Dokumente in opensky_raw: {len(df_osky)}\n")
print(df_osky.to_string(index=False))

Dokumente in opensky_raw: 4

       collected_at   endpoint airport  flight_count  window_h                      _id
2026-05-18T19:25:21   arrivals    EDDB           163        30 6a0b67a1957318837862d940
2026-05-18T19:25:20 departures    EDDB           166        30 6a0b67a0957318837862d93f
2026-05-18T19:22:37   arrivals    EDDB             0         6 6a0b66fd4f90cfa4d79ff9ba
2026-05-18T19:22:37 departures    EDDB            24         6 6a0b66fd4f90cfa4d79ff9b9


## 10. OpenSky Flüge aufklappen

Alle Flights aus allen opensky_raw-Dokumenten als flacher DataFrame.

In [11]:
rows = []
for doc in osky_col.find({}).sort("collected_at", 1):
    for f in doc.get("flights", []):
        rows.append({
            "icao24":    f.get("icao24", ""),
            "callsign":  (f.get("callsign") or "").strip(),
            "dep":       f.get("estDepartureAirport") or "?",
            "arr":       f.get("estArrivalAirport") or "?",
            "firstSeen": datetime.fromtimestamp(f["firstSeen"], timezone.utc).strftime("%Y-%m-%d %H:%M") if f.get("firstSeen") else None,
            "lastSeen":  datetime.fromtimestamp(f["lastSeen"],  timezone.utc).strftime("%H:%M")          if f.get("lastSeen")  else None,
            "endpoint":  doc["endpoint"],
            "airport":   doc["query_params"]["airport"],
        })

df_flights = pd.DataFrame(rows).drop_duplicates(subset=["icao24", "firstSeen"])
print(f"Unique Flüge (dedupliziert): {len(df_flights)}\n")
print(df_flights.head(20).to_string(index=False))

Unique Flüge (dedupliziert): 329

icao24 callsign  dep arr        firstSeen lastSeen   endpoint airport
4d00d7  LGL74BE EDDB   ? 2026-05-18 19:18    19:22 departures    EDDB
44029f  AUA238N EDDB   ? 2026-05-18 19:16    19:22 departures    EDDB
47878d  NOZ109B EDDB   ? 2026-05-18 19:13    19:22 departures    EDDB
39856d  AFR38LQ EDDB   ? 2026-05-18 19:05    19:22 departures    EDDB
46b82d   AEE82B EDDB   ? 2026-05-18 19:01    19:22 departures    EDDB
451e92   BGF004 EDDB   ? 2026-05-18 18:57    19:22 departures    EDDB
3c6751   DLH6EH EDDB   ? 2026-05-18 18:53    19:22 departures    EDDB
44cc43   BEL9NK EDDB   ? 2026-05-18 18:52    19:22 departures    EDDB
4b19f4  SWR979E EDDB   ? 2026-05-18 18:51    19:22 departures    EDDB
3c49cb    EWG3V EDDB   ? 2026-05-18 18:47    19:22 departures    EDDB
502d0e   BTI4UJ EDDB   ? 2026-05-18 18:45    19:22 departures    EDDB
4bc84c   PGT984 EDDB   ? 2026-05-18 18:23    19:22 departures    EDDB
407464   BAW995 EDDB   ? 2026-05-18 18:21    19:22 depar

## 11. Cross-Collection Join: ADS-B ↔ OpenSky

Join-Key: `adsb_raw.ac[].hex` = `opensky_raw.flights[].icao24` (ICAO24 Transponderadresse)

- **ADS-B** liefert: Echtzeit-Position, Höhe, Geschwindigkeit, Flugzeugtyp  
- **OpenSky** ergänzt: Abflug- und Zielflughafen, Callsign, Abflugzeit  

Einschränkung: Match nur möglich wenn ein Flugzeug im ADS-B-Snapshot sichtbar war **und** im OpenSky-Zeitfenster erfasst wurde.

In [12]:
# OpenSky-Index aufbauen: icao24 → beste verfügbare Fluginfo
opensky_index = {}
for doc in osky_col.find({}):
    for f in doc.get("flights", []):
        icao = (f.get("icao24") or "").lower()
        if icao and icao not in opensky_index:
            opensky_index[icao] = {
                "callsign_osky": (f.get("callsign") or "").strip(),
                "dep":           f.get("estDepartureAirport") or "?",
                "arr":           f.get("estArrivalAirport") or "?",
                "firstSeen":     datetime.fromtimestamp(f["firstSeen"], timezone.utc).strftime("%H:%M UTC") if f.get("firstSeen") else "?",
            }

# Neuesten ADS-B Snapshot joinen
latest_adsb = db.collection("adsb_raw").find_one({}, sort=[("collected_at", -1)])
snapshot_time = latest_adsb["collected_at"][:19]

matches = []
for ac in latest_adsb.get("ac", []):
    hex_id = (ac.get("hex") or "").lower()
    if hex_id in opensky_index:
        osky = opensky_index[hex_id]
        matches.append({
            "icao24":    hex_id,
            "callsign":  (ac.get("flight") or "").strip() or osky["callsign_osky"],
            "type":      ac.get("t", "?"),
            "alt_ft":    ac.get("alt_baro", "?"),
            "gs_kts":    round(ac.get("gs") or 0),
            "lat":       round(ac.get("lat", 0), 3) if ac.get("lat") else "?",
            "lon":       round(ac.get("lon", 0), 3) if ac.get("lon") else "?",
            "dep":       osky["dep"],
            "arr":       osky["arr"],
            "firstSeen": osky["firstSeen"],
        })

df_joined = pd.DataFrame(matches).sort_values("callsign")

print(f"ADS-B Snapshot: {snapshot_time}  ({latest_adsb['total']} aircraft)")
print(f"OpenSky icao24s im Index: {len(opensky_index)}")
print(f"Matches: {len(df_joined)}\n")
print(df_joined.to_string(index=False))

ADS-B Snapshot: 2026-05-27T16:20:13  (31 aircraft)
OpenSky icao24s im Index: 190
Matches: 2

icao24 callsign type  alt_ft  gs_kts    lat    lon  dep  arr firstSeen
502cd7   BTI8BX BCS3   34000     480 53.368 14.201 EDDB EETN 18:54 UTC
3c0936  SDR83EF A320   16475     357 52.562 12.312 LOWL EDDB 19:40 UTC


## 15. Verbindung schließen

---
## 12. flight_tracker_raw — Überblick

Jedes Dokument = eine getrackte Flugzeugposition (kein Snapshot-Array).  
Felder: `flight_label`, `hex`, `registration`, `aircraft_type`, `lat`, `lon`, `alt_baro`, `gs`, `squawk`.

In [13]:
ft_col = db.collection("flight_tracker_raw")

total_ft = ft_col.count_documents({})
latest_ft = ft_col.find_one({}, sort=[("collected_at", -1)])
oldest_ft = ft_col.find_one({}, sort=[("collected_at",  1)])
print(f"Dokumente gesamt: {total_ft}")
print(f"Ältester Eintrag: {oldest_ft['collected_at'][:19]}")
print(f"Neuester Eintrag: {latest_ft['collected_at'][:19]}")

docs_ft = list(ft_col.find({}))
df_ft = pd.DataFrame([
    {
        "collected_at":  d["collected_at"][:19],
        "flight_label":  d.get("flight_label", ""),
        "hex":           d.get("hex", ""),
        "registration":  d.get("registration", ""),
        "aircraft_type": d.get("aircraft_type", ""),
        "alt_baro":      d.get("alt_baro", ""),
        "gs":            d.get("gs", ""),
        "lat":           round(d["lat"], 3) if d.get("lat") else None,
        "lon":           round(d["lon"], 3) if d.get("lon") else None,
        "squawk":        d.get("squawk", ""),
    }
    for d in docs_ft
])
print(f"\nShape: {df_ft.shape}")
print(df_ft.head(10).to_string(index=False))

Dokumente gesamt: 116
Ältester Eintrag: 2026-05-20T18:37:15
Neuester Eintrag: 2026-05-20T19:35:37



Shape: (116, 10)


       collected_at flight_label    hex registration aircraft_type alt_baro    gs    lat    lon squawk
2026-05-20T18:37:15       EW7755 3c48ee       D-ABGN          A319    37025 421.2 49.743 14.863   5140
2026-05-20T18:37:45       EW7755 3c48ee       D-ABGN          A319    37200 421.9 49.795 14.815   5140
2026-05-20T18:38:15       EW7755 3c48ee       D-ABGN          A319    37675 424.0 49.844 14.771   5140
2026-05-20T18:38:45       EW7755 3c48ee       D-ABGN          A319    37975 420.9 49.896 14.724   5140
2026-05-20T18:39:15       EW7755 3c48ee       D-ABGN          A319    38025 413.7 49.946 14.678   5140
2026-05-20T18:39:46       EW7755 3c48ee       D-ABGN          A319    38025 414.2 49.995 14.634   5140
2026-05-20T18:40:16       EW7755 3c48ee       D-ABGN          A319    38475 411.3 50.044 14.586   5140
2026-05-20T18:40:46       EW7755 3c48ee       D-ABGN          A319    38950 411.9 50.092 14.537   5140
2026-05-20T18:41:16       EW7755 3c48ee       D-ABGN          A319    39

## 13. Aggregation — Flugzeugtypen und aktivste Flüge

Welche Typen tauchen am häufigsten auf? Welche `flight_label` haben die meisten Einträge?

In [14]:
# Flugzeugtypen
type_counts = df_ft["aircraft_type"].replace("", pd.NA).dropna().value_counts()
print("Flugzeugtypen:")
print(type_counts.to_string())

print()

# Aktivste Flüge (flight_label)
flight_counts = df_ft["flight_label"].replace("", pd.NA).dropna().value_counts().head(15)
print("Aktivste Flüge (flight_label, Top 15):")
print(flight_counts.to_string())

# MongoDB Aggregation zum Vergleich
pipeline = [
    {"$group": {"_id": "$aircraft_type", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
    {"$limit": 10},
]
print("\nMongoDB $group aircraft_type:")
for r in ft_col.aggregate(pipeline):
    print(f"  {r['_id'] or '(leer)':<10}  {r['count']:>4}")

Flugzeugtypen:
aircraft_type
A319    116

Aktivste Flüge (flight_label, Top 15):
flight_label
EW7755    116

MongoDB $group aircraft_type:


  A319         116


## 14. Positionsverlauf eines einzelnen Flugzeugs

Alle Einträge für ein bestimmtes `hex` (ICAO24) chronologisch sortiert.

In [15]:
# Häufigstes hex nehmen
top_hex = df_ft["hex"].value_counts().idxmax()
track_docs = list(ft_col.find({"hex": top_hex}).sort("collected_at", 1))

df_track = pd.DataFrame([
    {
        "collected_at": d["collected_at"][:19],
        "flight_label": d.get("flight_label", ""),
        "alt_baro":     d.get("alt_baro", ""),
        "gs":           d.get("gs", ""),
        "lat":          round(d["lat"], 4) if d.get("lat") else None,
        "lon":          round(d["lon"], 4) if d.get("lon") else None,
    }
    for d in track_docs
])

print(f"hex: {top_hex}  —  {len(df_track)} Einträge")
print(df_track.to_string(index=False))

hex: 3c48ee  —  116 Einträge
       collected_at flight_label alt_baro    gs     lat     lon
2026-05-20T18:37:15       EW7755    37025 421.2 49.7426 14.8626
2026-05-20T18:37:45       EW7755    37200 421.9 49.7947 14.8151
2026-05-20T18:38:15       EW7755    37675 424.0 49.8435 14.7710
2026-05-20T18:38:45       EW7755    37975 420.9 49.8957 14.7238
2026-05-20T18:39:15       EW7755    38025 413.7 49.9458 14.6782
2026-05-20T18:39:46       EW7755    38025 414.2 49.9946 14.6337
2026-05-20T18:40:16       EW7755    38475 411.3 50.0437 14.5861
2026-05-20T18:40:46       EW7755    38950 411.9 50.0924 14.5374
2026-05-20T18:41:16       EW7755    39000 417.8 50.1418 14.4882
2026-05-20T18:41:46       EW7755    39000 420.8 50.1904 14.4399
2026-05-20T18:42:17       EW7755    39000 420.5 50.2392 14.3913
2026-05-20T18:42:49       EW7755    38975 422.2 50.2919 14.3385
2026-05-20T18:43:19       EW7755    38975 418.3 50.3426 14.2878
2026-05-20T18:43:49       EW7755    38950 416.1 50.3906 14.2396
2026-05-20T

In [16]:
db.close()
print("Verbindung geschlossen")

Verbindung geschlossen
